### Initialization

In [0]:
%pip install databricks-vectorsearch
dbutils.library.restartPython()

In [0]:
from databricks.vector_search.client import VectorSearchClient

ENDPOINT_NAME = "humanoid_vs_endpoint"
INDEX_NAME    = "humanoid_robotics.silver.doc_chunks_index"

vsc   = VectorSearchClient(disable_notice=True)
index = vsc.get_index(endpoint_name=ENDPOINT_NAME, index_name=INDEX_NAME)

### Retrieval Check

In [0]:
def search(question: str, num_results: int = 5, filters: dict | None = None) -> list[dict]:
    """Query the index and return each hit as a dict (column name -> value, plus score)."""
    response = index.similarity_search(
        query_text=question,
        columns=["chunk_id", "doc_id", "title", "url", "publish_date", "company_ids", "chunk_text"],
        num_results=num_results,
        filters=filters,
    )
    column_names = [c["name"] for c in response["manifest"]["columns"]]
    return [dict(zip(column_names, row)) for row in response["result"]["data_array"]]


def show(hits: list[dict]) -> None:
    for h in hits:
        print(f"{h['score']:.3f}  {h['chunk_id']}  {h['title'][:70]}")
        print(f"       {h['chunk_text'][:160]!r}\n")